# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing Croissant schema entities by their `@id`.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs referenced via `@id`.

> **Note:** The Croissant schema metadata can be introspected for record set, field, and column `@id`s. Here, we show record sets, fields, and columns that are available in the dataset's schema.

In [ ]:
# Explore available record sets and fields by @id
schema_jsonld = dataset.metadata.to_json()

# Record sets may be referenced via 'recordSet' at root or in 'hasPart' or 'distribution'.
record_sets_ids = []
fields_by_record_set = {}

# Try to extract record sets (simulate Croissant structure)
if 'recordSet' in schema_jsonld and schema_jsonld['recordSet']:
    for rs in schema_jsonld['recordSet']:
        if isinstance(rs, dict):
            rs_id = rs.get('@id')
        else:
            rs_id = rs
        if rs_id:
            record_sets_ids.append(rs_id)
else:
    # Try from distribution
    if 'distribution' in schema_jsonld:
        for dist in schema_jsonld['distribution']:
            if isinstance(dist, dict) and dist.get('@id'):
                record_sets_ids.append(dist['@id'])
            elif isinstance(dist, str):
                record_sets_ids.append(dist)

# Show the list of record sets by @id
print('Record Sets (@id):')
for rs_id in record_sets_ids:
    print(f"  - {rs_id}")

# Try to print available fields for each record set (if present in schema)
fields_found = False
for rs_id in record_sets_ids:
    record_set_obj = None
    for key in ['recordSet', 'distribution', 'hasPart']:
        if key in schema_jsonld:
            for rset in schema_jsonld[key]:
                if isinstance(rset, dict) and rset.get('@id') == rs_id:
                    record_set_obj = rset
    if record_set_obj is None:
        continue
    if 'field' in record_set_obj:
        print(f"Fields for Record Set {rs_id}:")
        for fld in record_set_obj['field']:
            if isinstance(fld, dict):
                print(f"  - {fld.get('@id')} (label: {fld.get('name', '')})")
            else:
                print(f"  - {fld}")
        fields_found = True

if not fields_found:
    print("Fields information not found directly in schema.")

# Show a sample record from each record set (using mlcroissant generator)
for rs_id in record_sets_ids:
    print(f"\nSample record for record set '@id': {rs_id}")
    try:
        for idx, record in enumerate(dataset.records(record_set=rs_id)):
            print(json.dumps(record, indent=2))
            if idx >= 0:  # Show only one record for brevity
                break
    except Exception as e:
        print(f"Could not load records for {rs_id} (error: {e})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references use `@id`.

In [ ]:
# Extract records from each record set
dataframes = {}
for rs_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for record set '@id': {rs_id}")
            print(f"Columns: {df.columns.tolist()}")
            print(df.head())
        else:
            print(f"No records found for record set '@id': {rs_id}")
    except Exception as e:
        print(f"Could not extract records for {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps: filtering, normalization, grouping. We reference fields by their `@id`, which you can retrieve from the overview above or via DataFrame columns. Replace placeholder variable values with real `@id`s or column names as discovered.

In [ ]:
# Choose one record set to analyze.
# For demonstration, use the first record set with loaded data.
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Find a numeric field by column name or croissant @id (manual selection based on print above)
    numeric_candidates = [col for col in df.columns if ('age' in col.lower() or 'interval' in col.lower() or 'years' in col.lower())]
    numeric_field = numeric_candidates[0] if numeric_candidates else None

    print(f'Chosen numeric field for EDA: {numeric_field}')
    if numeric_field:
        threshold = 10  # Example threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Grouping (choose a group field; e.g., 'sex', 'msi_status', or similar)
        group_candidates = [col for col in df.columns if ('sex' in col.lower() or 'site' in col.lower() or 'msi' in col.lower())]
        group_field = group_candidates[0] if group_candidates else None
        print(f'Chosen group field for grouping: {group_field}')
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data ({numeric_field} mean) by {group_field}:")
            print(grouped_df.head())
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if dataframes and numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field], bins=16, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # Visualize grouping
    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR^2 dataset using `mlcroissant`, referencing all entities by their `@id`. You can extend this workflow by using more `@id` entities from the schema for advanced analysis and modeling.

- We loaded the dataset metadata and extracted available record sets and fields by their `@id`.
- Data was loaded into DataFrames from each record set for inspection.
- Typical EDA steps, including filtering, normalization, and grouping, were performed referencing fields by their `@id`.
- Visualizations illustrated key distributions and relationships.

Further steps could include handling additional record sets, integrating Croissant-compliant visualizations, or applying domain-specific modeling.
